# RailGuard AI â€” Notebook 02: YOLO11 Training & Progress Monitoring

This notebook analyzes the **YOLO11 Nano (`yolo11n.pt`) training progression** for RailGuard AI.

### Objectives:
1. Parse the training execution metrics from `ai-service/runs/yolo11_railguard_v1/results.csv`.
2. Plot Training Loss Trajectories (`box_loss`, `cls_loss`, `dfl_loss`).
3. Plot Validation Metric Learning Curves ($mAP_{50}$, $mAP_{50-95}$, Precision, Recall).
4. Evaluate learning stability and hyperparameter behavior across epochs.

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define paths
RUN_DIR = os.path.abspath("../ai-service/runs/yolo11_railguard_v1")
RESULTS_CSV = os.path.join(RUN_DIR, "results.csv")

print(f"Run Directory: {RUN_DIR}")
print(f"Results CSV: {RESULTS_CSV}")
print(f"Results Exists: {os.path.exists(RESULTS_CSV)}")

Run Directory: C:\Users\Ankadala Triveni\OneDrive\Documents\RailGuardAI\ai-service\runs\yolo11_railguard_v1
Results CSV: C:\Users\Ankadala Triveni\OneDrive\Documents\RailGuardAI\ai-service\runs\yolo11_railguard_v1\results.csv
Results Exists: True


## 1. Load Training Metrics (`results.csv`)

In [2]:
if os.path.exists(RESULTS_CSV):
    df_results = pd.read_csv(RESULTS_CSV)
    # Clean column names (Ultralytics strips leading/trailing whitespaces)
    df_results.columns = [c.strip() for c in df_results.columns]
    print(f"Loaded {len(df_results)} epochs of training data.")
    print(df_results.head())
else:
    print("Training in progress... results.csv will be available upon epoch completion.")

Loaded 12 epochs of training data.
   epoch       time  train/box_loss  train/cls_loss  train/dfl_loss  metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  metrics/mAP50-95(B)  val/box_loss  val/cls_loss  val/dfl_loss    lr/pg0    lr/pg1    lr/pg2
0      1    776.175         2.56650         5.46538         2.01484               0.72182            0.03826           0.01846              0.00456       2.49376       5.76064       1.92757  0.000174  0.000174  0.000174
1      2   3391.340         2.32181         4.42727         1.78815               0.58414            0.04699           0.03068              0.01169       2.48135       4.81047       2.03344  0.000343  0.000343  0.000343
2      3  10299.000         2.31618         3.98677         1.76580               0.25789            0.12073           0.03789              0.01171       2.54919       4.35039       1.89871  0.000504  0.000504  0.000504
3      4  11253.300         2.32650         3.79214         1.80746               0.2

## 2. Loss Trajectory Plots

In [3]:
if 'df_results' in locals() and not df_results.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Box Loss
    axes[0].plot(df_results['epoch'], df_results['train/box_loss'], label='Train Box Loss', color='blue')
    if 'val/box_loss' in df_results.columns:
        axes[0].plot(df_results['epoch'], df_results['val/box_loss'], label='Val Box Loss', color='orange', linestyle='--')
    axes[0].set_title("Bounding Box Loss (`box_loss`)", fontweight='bold')
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True)
    
    # Class Loss
    axes[1].plot(df_results['epoch'], df_results['train/cls_loss'], label='Train Class Loss', color='green')
    if 'val/cls_loss' in df_results.columns:
        axes[1].plot(df_results['epoch'], df_results['val/cls_loss'], label='Val Class Loss', color='red', linestyle='--')
    axes[1].set_title("Classification Loss (`cls_loss`)", fontweight='bold')
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True)
    
    # DFL Loss
    axes[2].plot(df_results['epoch'], df_results['train/dfl_loss'], label='Train DFL Loss', color='purple')
    if 'val/dfl_loss' in df_results.columns:
        axes[2].plot(df_results['epoch'], df_results['val/dfl_loss'], label='Val DFL Loss', color='brown', linestyle='--')
    axes[2].set_title("Distribution Focal Loss (`dfl_loss`)", fontweight='bold')
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Loss")
    axes[2].legend()
    axes[2].grid(True)
    
    plt.suptitle("RailGuard AI â€” Training Loss Curves Across Epochs", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Validation Metric Curves ($mAP_{50}$ and $mAP_{50-95}$)

In [4]:
if 'df_results' in locals() and not df_results.empty:
    plt.figure(figsize=(12, 6))
    if 'metrics/mAP50(B)' in df_results.columns:
        plt.plot(df_results['epoch'], df_results['metrics/mAP50(B)'], marker='o', label='mAP50', color='navy', linewidth=2)
    if 'metrics/mAP50-95(B)' in df_results.columns:
        plt.plot(df_results['epoch'], df_results['metrics/mAP50-95(B)'], marker='s', label='mAP50-95', color='crimson', linewidth=2)
        
    plt.title("RailGuard AI â€” Validation mAP Learning Trajectory", fontsize=14, fontweight='bold')
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("mAP Score", fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()